In [14]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


RESULTS_FOLDER = Path(
    "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/benchmark/downloads/results/bio/base_run"
)

In [15]:
records = []

for config_path in sorted(RESULTS_FOLDER.glob("*/seed_*/config.json")):
    config = json.loads(config_path.read_text())
    metrics = json.loads(config_path.with_name("metrics.json").read_text())

    model_type = config["predictor_config"]["type"]
    if config["rearrangement_config"] is not None:
        model_type += "_rearranged"

    calibrator = config["conformal_config"]["calibrator"]
    conformal_score_type = calibrator["type"]
    if conformal_score_type == "norm":
        conformal_score_type = f"l{calibrator['p']:g}"

    for metric_name, metric in metrics.items():
        if isinstance(metric, dict):
            records.append(
                {
                    "model_type": model_type,
                    "conformal_score_type": conformal_score_type,
                    "seed": config["seed"],
                    "metric": metric_name,
                    "seed_mean": metric["mean"],
                }
            )

results = pd.DataFrame(records)
results = results.loc[
    (results["metric"] != "log_volume_per_dimension")
    | np.isfinite(results["seed_mean"])
]
statistics = (
    results.groupby(["model_type", "conformal_score_type", "metric"])["seed_mean"]
    .agg(mean="mean", std="std")
)

In [16]:
for (model_type, conformal_score_type), table in statistics.groupby(
    level=["model_type", "conformal_score_type"]
):
    print("=" * 80)
    print(f"Model: {model_type}")
    print(f"Conformal score: {conformal_score_type}\n")
    print(
        table.droplevel(["model_type", "conformal_score_type"])
        .rename(
            columns={
                "mean": "Mean across seeds",
                "std": "Std of seed means",
            }
        )
        .to_string(float_format=lambda value: f"{value:.6f}")
    )
    print()

Model: neural_optimal_transport
Conformal score: l2

                          Mean across seeds  Std of seed means
metric                                                        
excess_coverage_risk               0.063124           0.002923
log_volume_per_dimension           4.575632           0.006532
marginal_coverage                  0.901159           0.001748
worst_slab_coverage                0.850012           0.047878

Model: neural_optimal_transport_rearranged
Conformal score: l2

                          Mean across seeds  Std of seed means
metric                                                        
excess_coverage_risk               0.063291           0.001483
log_volume_per_dimension           4.540579           0.012065
marginal_coverage                  0.902318           0.001192
worst_slab_coverage                0.846418           0.035974

Model: normalizing_flow
Conformal score: l2

                          Mean across seeds  Std of seed means
metric           